## Smartwatch Rating Prediction: Hypertunning
- Searches over **regularization-focused** grids — the goal is not the highest possible train score, it's the smallest gap between train and test performance while keeping test R² competitive. On a ~350-row dataset, an unregularized tree ensemble will always overfit; these grids are deliberately biased toward simpler models (shallower trees, more samples per leaf, subsampling, L1/L2 penalties on boosting).

In [7]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor

import dagshub
dagshub.init(repo_owner='kushneek', repo_name='smartwatch-rating-prediction', mlflow=True)

import mlflow
import mlflow.sklearn
mlflow.set_experiment("smartwatch-rating")


TUNED_DIR = Path.cwd().parent / "experiments" / "tuned_models"
TUNED_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path.cwd().parent / "Dataset"
RESULTS_DIR = Path.cwd().parent / "results"

Initialized MLflow to track repo "kushneek/smartwatch-rating-prediction"

Repository kushneek/smartwatch-rating-prediction initialized!

## Load Data

In [8]:
# Importing Data

df = pd.read_csv(DATA_DIR / "Processed" / "cleaned_dataset.csv")
X_train = pd.read_csv(DATA_DIR / "split" / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "split" / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "split" / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "split" / "y_test.csv").squeeze()

# Scaled Data
X_train_scaled = pd.read_csv(DATA_DIR / "split" / "X_train_scaled.csv")
X_test_scaled = pd.read_csv(DATA_DIR / "split" / "X_test_scaled.csv")

### Parameter grid

In [9]:
param_grids = {
    "Ridge": {"alpha": [1, 5, 10, 25, 50, 100, 200]},
    "Lasso": {"alpha": [0.001, 0.005, 0.01, 0.05, 0.1]},
    "KNN": {
        "n_neighbors": [5, 7, 9, 11],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "Decision Tree": {
        "max_depth": [3, 4, 5, 6],
        "min_samples_leaf": [4, 8, 12],
        "min_samples_split": [10, 20]
    },
    "Random Forest": {
        "n_estimators": [200, 300],
        "max_depth": [4, 6, 8],
        "min_samples_leaf": [4, 8],
        "max_features": ["sqrt"]
    },
    "Gradient Boosting": {
        "n_estimators": [100, 150],
        "learning_rate": [0.03, 0.05],
        "max_depth": [2, 3],
        "subsample": [0.8],
        "min_samples_leaf": [4, 8]
    },
    "XGBoost": {
        "n_estimators": [100, 150],
        "learning_rate": [0.03, 0.05],
        "max_depth": [2, 3],
        "subsample": [0.8],
        "colsample_bytree": [0.8],
        "reg_alpha": [0, 1],
        "reg_lambda": [1, 5]
    },
}

In [10]:
scaled_models = ["Ridge","Lasso","KNN"]

models = {

    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "Ridge": Ridge(),

    "Lasso": Lasso(),

    "KNN": KNeighborsRegressor(),

    "Random Forest": RandomForestRegressor(random_state=42),

    "Gradient Boosting": GradientBoostingRegressor(random_state=42),

    "XGBoost": XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    )

}

In [11]:
# Removing any [, ], <, >, =

for col in X_train.columns:
    if "[" in col or "]"in col or "<" in col or  "=" in col:
        print(col)
        X_train.columns=(X_train.columns
            .str.replace("[", "", regex=False)
            .str.replace("]", "", regex=False)
            .str.replace("<", "less than",  regex=False)
            .str.replace(">", "greater than", regex=False)
            .str.replace("=", "equal to", regex=False)
        )
        X_test.columns=(X_test.columns
            .str.replace("[", "", regex=False)
            .str.replace("]", "", regex=False)
            .str.replace("<", "less than", regex=False)
            .str.replace(">", "greater than", regex=False)
            .str.replace("=", "equal to", regex=False)
        )

## Run GridSearchCV

In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_models = {}
results = []

for name, model in models.items():
    print("=" * 60)
    print(f"Tuning {name}")

    X_used = X_train_scaled if name in scaled_models else X_train

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring="r2",
        cv=kf,
        n_jobs=-1,
    )
    grid.fit(X_used, y_train)

    best_models[name] = grid.best_estimator_

    # Train/test gap for the tuned model, using the best CV params
    X_te = X_test_scaled if name in scaled_models else X_test
    train_r2 = grid.best_estimator_.score(X_used, y_train)
    test_r2 = grid.best_estimator_.score(X_te, y_test)

    with mlflow.start_run(run_name=name):
        mlflow.log_params(grid.best_params_)
        mlflow.log_metric("cv_r2_mean", grid.best_score_)
        mlflow.log_metric("train_r2", train_r2)
        mlflow.log_metric("test_r2", test_r2)
        mlflow.log_metric("cv_gap", train_r2 - grid.best_score_)
        mlflow.sklearn.log_model(grid.best_estimator_, name="model")


    results.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "Best CV R2": grid.best_score_,
        "Train R2": train_r2,
        "Test R2": test_r2,
        "Train-Test Gap": train_r2 - test_r2,
    })

    print("Best params:", grid.best_params_)
    print(f"CV R2: {grid.best_score_:.4f}  \nTrain R2: {train_r2:.4f}  \nTest R2: {test_r2:.4f}  \nGap: {train_r2 - test_r2:.4f}")

tuning_results_df = pd.DataFrame(results).sort_values("Test R2", ascending=False)
tuning_results_df

Tuning Decision Tree


2026/08/02 23:17:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Decision Tree at: https://dagshub.com/kushneek/smartwatch-rating-prediction.mlflow/#/experiments/0/runs/dae6cd89ffe049d0bc72709e0d64770e
🧪 View experiment at: https://dagshub.com/kushneek/smartwatch-rating-prediction.mlflow/#/experiments/0
Best params: {'max_depth': 4, 'min_samples_leaf': 4, 'min_samples_split': 20}
CV R2: 0.1140  
Train R2: 0.3964  
Test R2: 0.4352  
Gap: -0.0388
Tuning Ridge


2026/08/02 23:17:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Ridge at: https://dagshub.com/kushneek/smartwatch-rating-prediction.mlflow/#/experiments/0/runs/ca5a61ee61fe456b9b92e809cd5bbeac
🧪 View experiment at: https://dagshub.com/kushneek/smartwatch-rating-prediction.mlflow/#/experiments/0
Best params: {'alpha': 100}
CV R2: 0.1599  
Train R2: 0.2531  
Test R2: 0.3406  
Gap: -0.0874
Tuning Lasso


...

2026/08/02 23:19:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Gradient Boosting at: https://dagshub.com/kushneek/smartwatch-rating-prediction.mlflow/#/experiments/0/runs/a55c02d4489540d2a3cc161f9103c78f
🧪 View experiment at: https://dagshub.com/kushneek/smartwatch-rating-prediction.mlflow/#/experiments/0
Best params: {'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 4, 'n_estimators': 150, 'subsample': 0.8}
CV R2: 0.1288  
Train R2: 0.5072  
Test R2: 0.2846  
Gap: 0.2226
Tuning XGBoost


2026/08/02 23:20:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost at: https://dagshub.com/kushneek/smartwatch-rating-prediction.mlflow/#/experiments/0/runs/73ad28fa43174db9a258bd322a5de614
🧪 View experiment at: https://dagshub.com/kushneek/smartwatch-rating-prediction.mlflow/#/experiments/0
Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 150, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 0.8}
CV R2: 0.2326  
Train R2: 0.7046  
Test R2: 0.0273  
Gap: 0.6773


,Model,Best Parameters,Best CV R2,Train R2,Test R2,Train-Test Gap
0,Decision Tree,"{'max_depth': 4, 'min_samples_leaf': 4, 'min_s...",0.114048,0.396401,0.435154,-0.038753
2,Lasso,{'alpha': 0.01},0.174329,0.265998,0.357436,-0.091438
4,Random Forest,"{'max_depth': 8, 'max_features': 'sqrt', 'min_...",0.163013,0.475292,0.342651,0.132640
1,Ridge,{'alpha': 100},0.159897,0.253133,0.340583,-0.087450
5,Gradient Boosting,"{'learning_rate': 0.03, 'max_depth': 2, 'min_s...",0.128765,0.507223,0.284613,0.222610
3,KNN,"{'metric': 'euclidean', 'n_neighbors': 7, 'wei...",0.062848,0.288696,0.263897,0.024800
6,XGBoost,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.232572,0.704587,0.027288,0.677298


In [13]:
results

[{'Model': 'Decision Tree',
  'Best Parameters': {'max_depth': 4,
   'min_samples_leaf': 4,
   'min_samples_split': 20},
  'Best CV R2': np.float64(0.11404846826094994),
  'Train R2': 0.3964007857830547,
  'Test R2': 0.4351536529228246,
  'Train-Test Gap': -0.038752867139769887},
 {'Model': 'Ridge',
  'Best Parameters': {'alpha': 100},
  'Best CV R2': np.float64(0.15989706157357814),
  'Train R2': 0.25313345726502456,
  'Test R2': 0.340583069748449,
  'Train-Test Gap': -0.08744961248342442},
 {'Model': 'Lasso',
  'Best Parameters': {'alpha': 0.01},
  'Best CV R2': np.float64(0.17432867385533704),
  'Train R2': 0.2659982622391954,
  'Test R2': 0.3574362863757662,
  'Train-Test Gap': -0.09143802413657076},
 {'Model': 'KNN',
  'Best Parameters': {'metric': 'euclidean',
   'n_neighbors': 7,
   'weights': 'uniform'},
  'Best CV R2': np.float64(0.06284760426094946),
  'Train R2': 0.28869625790817377,
  'Test R2': 0.26389667741483247,
  'Train-Test Gap': 0.0247995804933413},
 {'Model': 'Rando

In [14]:
best_models

{'Decision Tree': DecisionTreeRegressor(max_depth=4, min_samples_leaf=4, min_samples_split=20,
                       random_state=42),
 'Ridge': Ridge(alpha=100),
 'Lasso': Lasso(alpha=0.01),
 'KNN': KNeighborsRegressor(metric='euclidean', n_neighbors=7),
 'Random Forest': RandomForestRegressor(max_depth=8, max_features='sqrt', min_samples_leaf=4,
                       n_estimators=300, random_state=42),
 'Gradient Boosting': GradientBoostingRegressor(learning_rate=0.03, max_depth=2, min_samples_leaf=4,
                           n_estimators=150, random_state=42, subsample=0.8),
 'XGBoost': XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constrain

In [15]:
import os

for name, model in best_models.items():
    filename = name.replace(" ", "_") + ".joblib"
    joblib.dump(model, TUNED_DIR / filename)

In [16]:
tuning_results_df.to_csv(RESULTS_DIR / "Hyperparameter_Tuning_Results.csv", index=False)